# 01 Project Overview

This notebook explains the MetroPT-3 compressor failure-risk prediction project at a high level. It is for review and explanation only.

The authoritative reproducible pipeline is in `src/`. Run project commands from the project root, not from inside `notebooks/`.

## Project Goal

The project is a supervised binary classification workflow for predictive maintenance of a metro train compressor.

Each example is a 1-minute compressor sensor window. The model predicts whether that window belongs to normal operation or a failure-risk/anomaly period.

## Target Labels

The binary target is:

- `0` = normal operation
- `1` = failure-risk/anomaly

A 1-minute window is labeled `1` if its timestamp occurs from 1 hour before a documented failure start time through the documented failure end time. This creates an early-warning failure-risk label rather than only labeling the failure after it has already started.

## Dataset

The dataset is the MetroPT-3 air compressor dataset from the UCI Machine Learning Repository. It contains timestamped compressor sensor readings such as pressure, temperature, current, and operating-state signals.

The raw CSV is kept locally under `data/raw/` and is not committed. The generated processed windowed file is `data/processed/windowed_labeled_data.csv`, which is also a generated local artifact and should not be committed.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "src").exists(), "Run this notebook from the project root or notebooks/."

for relative_path in [
    "src/preprocessing.py",
    "src/train.py",
    "src/evaluate.py",
    "results/eda_summary.csv",
    "results/test_metrics.csv",
]:
    path = PROJECT_ROOT / relative_path
    print(f"{relative_path}: {'found' if path.exists() else 'missing'}")

## Why Compressor Failure-Risk Prediction Matters

Metro train compressors support important operational functions. Compressor failures can create repair costs, service disruption, and operational risk. In this project, false negatives are especially important because a missed failure-risk window can mean a missed opportunity to inspect or repair the compressor before or during a failure period.

For that reason, the project emphasizes recall, F2-score, F1-score, precision, and the confusion matrix. Accuracy is reported, but it is not the main metric because the dataset is highly imbalanced.

## Evaluation Framing

The main realistic evaluation is an event-aware blocked split:

- Training: failure events 1 and 2
- Validation: failure event 3
- Test: failure event 4

This split is used because the positive windows come from only four independent documented failure events. Randomly mixing windows from the same failure events across train and test would give an overly optimistic view of generalization.

The stratified window-level baseline is included only as an optimistic comparison. It checks feature separability when event independence is not enforced, but it is not deployment-level performance.

## Current Main Finding

The selected final model is the Logistic Regression baseline with validation-selected threshold `0.61`.

The event-aware test on held-out failure event 4 failed to generalize. The final test result has `TP = 0`, `FN = 330`, recall `0.0`, F1 `0.0`, and F2 `0.0` for the failure-risk class. The current model is not deployment-ready.